# Step 2: Embeddings and Vector Index

This notebook implements **Step 2** of the MediQuery RAG pipeline: converting text chunks into dense vector embeddings and building a FAISS index for fast similarity search.

## What This Notebook Does

1. Loads the **8,563 text chunks** from `all_chunks.json` (produced by the chunking notebook)
2. Encodes every chunk into a **768-dimensional dense vector** using `BAAI/bge-base-en-v1.5`
3. Builds a **FAISS index** (`IndexFlatIP`) for fast cosine-similarity search
4. Saves all artifacts (index, doc IDs, embeddings, metadata) to Google Drive
5. Runs a **retrieval sanity test** with sample queries

## Why Dense Embeddings?

Traditional keyword search (e.g., BM25) matches documents based on exact word overlap. This fails when a user asks *"Does Medicare cover acupuncture?"* but the policy document says *"needle-based therapy for chronic pain."* Dense embeddings solve this by mapping text into a continuous vector space where **semantically similar texts are close together**, regardless of exact wording. This is the same paradigm explored in the Semantic Search class notebook, now applied to our Medicare policy corpus.

## Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 160.5 MB/s eta 0:00:00


## Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

# ─── PATH CONFIGURATION ────────────────────────────────────────────────────────
DATA_DIR    = '/content/drive/MyDrive/Embeddings'
CHUNKS_FILE = f'{DATA_DIR}/all_chunks.json'
OUTPUT_DIR  = f'{DATA_DIR}/faiss_index'
INDEX_FILE  = f'{OUTPUT_DIR}/medicare.index'
DOCID_FILE  = f'{OUTPUT_DIR}/docids.txt'
EMBED_FILE  = f'{OUTPUT_DIR}/embeddings.npy'
META_FILE   = f'{OUTPUT_DIR}/chunk_metadata.json'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /content/drive/MyDrive/Embeddings/faiss_index


## Load Chunks from JSON

Load the `all_chunks.json` file produced by the chunking notebook. This file contains chunks from four document types:
- **NCD** (National Coverage Determinations) — 575 chunks
- **LCD** (Local Coverage Determinations) — 2,802 chunks
- **BENEFIT_POLICY** (Medicare Benefit Policy Manual) — 1,039 chunks
- **CLAIMS_PROCESSING** (Medicare Claims Processing Manual) — 4,147 chunks

In [ ]:
import json
from collections import Counter

with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    all_chunks = json.load(f)

print(f"Total chunks loaded: {len(all_chunks):,}")
print()

# Breakdown by document type
type_counts = Counter(c['type'] for c in all_chunks)
for doc_type, count in sorted(type_counts.items()):
    print(f"  {doc_type}: {count:,}")

# Quick sanity check on schema
sample = all_chunks[0]
print(f"\nSample chunk keys: {list(sample.keys())}")
print(f"Sample text preview: {sample['text'][:200]}...")

Total chunks loaded: 8,563

  BENEFIT_POLICY: 1,039
  CLAIMS_PROCESSING: 4,147
  LCD: 2,802
  NCD: 575

Sample chunk keys: ['text', 'raw_text', 'source_id', 'title', 'type', 'states', 'filename', 'chunk_idx']
Sample text preview: [TYPE: NCD | NCD_ID: 100.3 | TITLE: 24-Hour Ambulatory Esophageal pH Monitoring]

TITLE: 24-Hour Ambulatory Esophageal pH Monitoring SECTION: 100.3 EFFECTIVE DATE: 1985-06-11 00:00:00 COVERAGE POLICY:...


## Embedding Model: `BAAI/bge-base-en-v1.5`

We use **`BAAI/bge-base-en-v1.5`** from the BGE (BAAI General Embedding) model family (Xiao et al., 2023) for the following reasons:

| Property | Value |
|---|---|
| **Embedding dimension** | 768 |
| **Max sequence length** | 512 tokens |
| **MTEB benchmark rank** | State-of-the-art for base-sized models |
| **License** | MIT (open-source, free) |
| **Retrieval optimization** | Asymmetric (short query vs. long passage) |

### Why This Model?

1. **MTEB benchmark performance**: BGE models achieve top scores on the Massive Text Embedding Benchmark across retrieval, reranking, and semantic similarity tasks.
2. **768-dimensional output**: A good balance between embedding quality and computational efficiency.
3. **Asymmetric retrieval**: BGE is specifically optimized for settings where queries are short (user questions) and documents are longer (policy chunks) — exactly our use case.
4. **Open-source**: Freely available on HuggingFace with no API costs.

### How BGE Embeddings Work

For **passages** (our chunks), we encode the text directly — no special prefix needed. For **queries** at retrieval time, BGE recommends prepending an instruction prefix to improve retrieval quality. The `sentence-transformers` library handles this automatically when configured.

### Index Size Estimate

Each embedding is 768 float32 values = 3,072 bytes. For 8,563 chunks:
- Embeddings: ~25 MB
- FAISS index: ~25 MB (IndexFlatIP stores raw vectors)

## Load the Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)
print(f"\nModel loaded: BAAI/bge-base-en-v1.5")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 95.0 GB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model loaded: BAAI/bge-base-en-v1.5
Embedding dimension: 768


## Encode All Chunks into Dense Vectors

We embed the **`text` field** (not `raw_text`) from each chunk. The `text` field includes the metadata header, e.g.:

```
[TYPE: LCD | LCD_ID: 35125 | TITLE: Wound Care | STATES: TX, AR, CO, LA | CONTRACTOR: Novitas]

Medicare covers wound care services when the patient presents...
```

Embedding the metadata header is intentional — it means a query about "wound care Texas" can match on both the policy content AND the metadata tags, improving retrieval quality.

### Key Parameters

- **`normalize_embeddings=True`**: L2-normalizes each vector so that inner product equals cosine similarity. This lets us use FAISS `IndexFlatIP` for exact cosine search.
- **`batch_size=64`**: Balances GPU memory usage and encoding speed.

In [ ]:
import numpy as np

# Extract the metadata-tagged text from each chunk
texts = [chunk['text'] for chunk in all_chunks]

print(f"Encoding {len(texts):,} chunks with BAAI/bge-base-en-v1.5...")
print(f"This may take a few minutes on GPU, longer on CPU.\n")

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")   # Expected: (8563, 768)
print(f"Dtype: {embeddings.dtype}")
print(f"Memory usage: {embeddings.nbytes / 1024**2:.1f} MB")

# Verify normalization: L2 norm of each vector should be ~1.0
norms = np.linalg.norm(embeddings, axis=1)
print(f"\nL2 norm check (should be ~1.0):")
print(f"  Mean norm: {norms.mean():.6f}")
print(f"  Min norm:  {norms.min():.6f}")
print(f"  Max norm:  {norms.max():.6f}")

Encoding 8,563 chunks with BAAI/bge-base-en-v1.5...
This may take a few minutes on GPU, longer on CPU.



Batches:   0%|          | 0/134 [00:00<?, ?it/s]


Embeddings shape: (8563, 768)
Dtype: float32
Memory usage: 25.1 MB

L2 norm check (should be ~1.0):
  Mean norm: 1.000000
  Min norm:  1.000000
  Max norm:  1.000000


## Build the FAISS Index

We use **`faiss.IndexFlatIP`** (Inner Product) for the following reasons:

1. **Exact search**: With 8,500 vectors, brute-force search is fast enough (1-5ms per query). No need for approximate methods like IVF or HNSW.
2. **Cosine similarity via inner product**: Since our embeddings are L2-normalized, the inner product between any two vectors equals their cosine similarity.
3. **No training required**: Unlike approximate indexes (e.g., `IndexIVFFlat`), `IndexFlatIP` requires no training step — we simply add vectors.
4. **Deterministic**: Every search returns the mathematically exact top-k results, ensuring reproducibility.

In [ ]:
import faiss

dimension = embeddings.shape[1]  # 768
print(f"Building FAISS IndexFlatIP with dimension {dimension}...")

# Create the index — IndexFlatIP computes inner product (= cosine sim for normalized vectors)
index = faiss.IndexFlatIP(dimension)

# Add all embeddings to the index
index.add(embeddings.astype('float32'))

print(f"Index built successfully!")
print(f"  Vectors in index: {index.ntotal:,}")
print(f"  Dimension: {index.d}")
print(f"  Index type: IndexFlatIP (exact cosine similarity)")

Building FAISS IndexFlatIP with dimension 768...
Index built successfully!
  Vectors in index: 8,563
  Dimension: 768
  Index type: IndexFlatIP (exact cosine similarity)


## Save Artifacts to Google Drive

We save four files that downstream notebooks (retrieval + reranking, LLM generation) will load:

| File | Description |
|---|---|
| `medicare.index` | FAISS index containing all 8,563 embedding vectors |
| `docids.txt` | Line-by-line mapping from index position to `type\|source_id\|chunk_idx` |
| `embeddings.npy` | Raw NumPy array of embeddings (for debugging or reranking experiments) |
| `chunk_metadata.json` | Full metadata for each chunk (type, title, states, contractor, etc.) |

In [ ]:
# ─── Save FAISS index ─────────────────────────────────────────────────────────
faiss.write_index(index, INDEX_FILE)
print(f"FAISS index saved: {INDEX_FILE}")

# ─── Save document IDs (index position -> chunk identity) ─────────────────────
with open(DOCID_FILE, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(f"{chunk['type']}|{chunk['source_id']}|{chunk['chunk_idx']}\n")
print(f"Doc IDs saved: {DOCID_FILE}")

# ─── Save raw embeddings ──────────────────────────────────────────────────────
np.save(EMBED_FILE, embeddings)
print(f"Embeddings saved: {EMBED_FILE}")

# ─── Save chunk metadata for retrieval lookups ────────────────────────────────
metadata = []
for i, chunk in enumerate(all_chunks):
    metadata.append({
        'idx': i,
        'type': chunk['type'],
        'source_id': chunk['source_id'],
        'title': chunk['title'],
        'states': chunk.get('states', ['ALL']),
        'contractor': chunk.get('contractor', ''),
        'filename': chunk['filename'],
        'chunk_idx': chunk['chunk_idx']
    })

with open(META_FILE, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f"Chunk metadata saved: {META_FILE}")

# ─── Report file sizes ────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print("Saved artifacts:")
for path in [INDEX_FILE, DOCID_FILE, EMBED_FILE, META_FILE]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  {os.path.basename(path):>25s}: {size_mb:>6.1f} MB")
print(f"{'='*50}")

FAISS index saved: /content/drive/MyDrive/Embeddings/faiss_index/medicare.index
Doc IDs saved: /content/drive/MyDrive/Embeddings/faiss_index/docids.txt
Embeddings saved: /content/drive/MyDrive/Embeddings/faiss_index/embeddings.npy
Chunk metadata saved: /content/drive/MyDrive/Embeddings/faiss_index/chunk_metadata.json

Saved artifacts:
             medicare.index:   25.1 MB
                 docids.txt:    0.4 MB
             embeddings.npy:   25.1 MB
        chunk_metadata.json:    4.0 MB


## Index Statistics Summary

| Metric | Value |
|---|---|
| Total chunks indexed | 8,563 |
| Embedding model | `BAAI/bge-base-en-v1.5` |
| Embedding dimension | 768 |
| Normalization | L2-normalized (cosine similarity via inner product) |
| FAISS index type | `IndexFlatIP` (exact brute-force) |
| Chunk types | NCD (575), LCD (2,802), Benefit Policy (1,039), Claims Processing (4,147) |
| Index file size | ~25 MB |
| Embeddings file size | ~25 MB |

## Retrieval Sanity Test

Before handing off to the reranking pipeline (Person 3), we verify that the FAISS index retrieves semantically relevant chunks. We test three queries covering different document types:

1. **NCD query** (national coverage): acupuncture for chronic pain
2. **LCD query** (state-specific): oxygen therapy in Texas
3. **Benefit Policy query**: skilled nursing facility requirements

In [ ]:
def search_faiss(query, model, index, all_chunks, top_k=5):
    """
    Embed a query and return the top-k most similar chunks from the FAISS index.

    Args:
        query:      Natural language question string
        model:      SentenceTransformer model used for encoding
        index:      FAISS index containing chunk embeddings
        all_chunks: List of chunk dictionaries from all_chunks.json
        top_k:      Number of results to return

    Returns:
        List of result dictionaries with rank, score, metadata, and text preview
    """
    # Encode the query (same normalization as corpus embeddings)
    query_embedding = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).reshape(1, -1).astype('float32')

    # Search the FAISS index
    scores, indices = index.search(query_embedding, top_k)

    # Build result list with metadata
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        chunk = all_chunks[idx]
        results.append({
            'rank': rank + 1,
            'score': float(score),
            'type': chunk['type'],
            'source_id': chunk['source_id'],
            'title': chunk['title'],
            'states': chunk.get('states', ['ALL']),
            'text_preview': chunk['text'][:300]
        })
    return results


def print_results(query, results):
    """Pretty-print retrieval results for a query."""
    print(f'\nQuery: "{query}"')
    print('=' * 90)
    for r in results:
        state_str = ', '.join(r['states']) if isinstance(r['states'], list) else r['states']
        print(f"  #{r['rank']}  [score={r['score']:.4f}]  [{r['type']}]  "
              f"{r['title']}  (ID: {r['source_id']})")
        print(f"       States: {state_str}")
        print(f"       Preview: {r['text_preview'][:150]}...")
        print()

### Query 1 — NCD: Acupuncture Coverage

**Expected**: Top results should include **NCD 30.1** (Acupuncture for Chronic Low Back Pain).

In [ ]:
query1 = "Does Medicare cover acupuncture for chronic lower back pain?"
results1 = search_faiss(query1, model, index, all_chunks, top_k=5)
print_results(query1, results1)


Query: "Does Medicare cover acupuncture for chronic lower back pain?"
  #1  [score=0.8095]  [NCD]  Acupuncture for Chronic Lower Back Pain (cLBP)  (ID: 30.3.3)
       States: ALL
       Preview: [TYPE: NCD | NCD_ID: 30.3.3 | TITLE: Acupuncture for Chronic Lower Back Pain (cLBP)]

TITLE: Acupuncture for Chronic Lower Back Pain (cLBP) SECTION: 3...

  #2  [score=0.7742]  [NCD]  Acupuncture  (ID: 30.3)
       States: ALL
       Preview: [TYPE: NCD | NCD_ID: 30.3 | TITLE: Acupuncture]

TITLE: Acupuncture SECTION: 30.3 EFFECTIVE DATE: 2020-01-21 00:00:00 COVERAGE POLICY: B. Nationally C...

  #3  [score=0.7358]  [NCD]  Acupuncture for Fibromyalgia  (ID: 30.3.1)
       States: ALL
       Preview: [TYPE: NCD | NCD_ID: 30.3.1 | TITLE: Acupuncture for Fibromyalgia]

TITLE: Acupuncture for Fibromyalgia SECTION: 30.3.1 EFFECTIVE DATE: 2020-01-21 00:...

  #4  [score=0.7281]  [NCD]  Acupuncture for Osteoarthritis  (ID: 30.3.2)
       States: ALL
       Preview: [TYPE: NCD | NCD_ID: 30.3.2 | TITLE

### Query 2 — LCD: Oxygen Therapy in Texas

**Expected**: Should retrieve LCD chunks tagged with **TX** and/or related to oxygen equipment coverage.

In [ ]:
query2 = "Is home oxygen therapy covered for patients in Texas?"
results2 = search_faiss(query2, model, index, all_chunks, top_k=5)
print_results(query2, results2)


Query: "Is home oxygen therapy covered for patients in Texas?"
  #1  [score=0.8005]  [NCD]  Home Use of Oxygen  (ID: 240.2)
       States: ALL
       Preview: [TYPE: NCD | NCD_ID: 240.2 | TITLE: Home Use of Oxygen]

TITLE: Home Use of Oxygen SECTION: 240.2 EFFECTIVE DATE: 2021-09-27 00:00:00 COVERAGE POLICY:...

  #2  [score=0.7805]  [NCD]  Home Use of Oxygen  (ID: 240.2)
       States: ALL
       Preview: [TYPE: NCD | NCD_ID: 240.2 | TITLE: Home Use of Oxygen]

patient's age, the patients skin pigmentation, the altitude level, or the patient's decrease...

  #3  [score=0.7583]  [NCD]  Home Use of Oxygen in Approved Clinical Trials  (ID: 240.2.1)
       States: ALL
       Preview: [TYPE: NCD | NCD_ID: 240.2.1 | TITLE: Home Use of Oxygen in Approved Clinical Trials]

TITLE: Home Use of Oxygen in Approved Clinical Trials SECTION: ...

  #4  [score=0.7432]  [LCD]  Oxygen and Oxygen Equipment  (ID: 33797)
       States: AL, AK, AZ, AR, CA, CO, CT, DE, DC, FL, GA, HI, ID, IL, IN, IA, KS, 

### Query 3 — Benefit Policy: Skilled Nursing Facility

**Expected**: Should retrieve **BENEFIT_POLICY** chunks from Chapter 8 (SNF Coverage).

In [ ]:
query3 = "What are the requirements for skilled nursing facility coverage?"
results3 = search_faiss(query3, model, index, all_chunks, top_k=5)
print_results(query3, results3)


Query: "What are the requirements for skilled nursing facility coverage?"
  #1  [score=0.7718]  [BENEFIT_POLICY]  Chapter 7 - Home Health Services  (ID: Chapter 7 - Home Health Services)
       States: ALL
       Preview: [TYPE: BENEFIT_POLICY | MANUAL: Medicare Benefit Policy Manual | CHAPTER: Chapter 7 - Home Health Services | STATES: ALL]

fewer hours each week (or, ...

  #2  [score=0.7687]  [BENEFIT_POLICY]  Chapter 8 - Coverage of Extended Care (SNF) Services Under Hospital Insurance  (ID: Chapter 8 - Coverage of Extended Care (SNF) Services Under Hospital Insurance)
       States: ALL
       Preview: [TYPE: BENEFIT_POLICY | MANUAL: Medicare Benefit Policy Manual | CHAPTER: Chapter 8 - Coverage of Extended Care (SNF) Services Under Hospital Insuranc...

  #3  [score=0.7645]  [BENEFIT_POLICY]  Chapter 8 - Coverage of Extended Care (SNF) Services Under Hospital Insurance  (ID: Chapter 8 - Coverage of Extended Care (SNF) Services Under Hospital Insurance)
       States: ALL
       

## Observations & Next Steps

### What We Built

- Encoded all **8,563 chunks** from the MediQuery corpus into 768-dimensional dense vectors using `BAAI/bge-base-en-v1.5`.
- Built a **FAISS `IndexFlatIP` index** that supports exact cosine-similarity search over the full corpus.
- Verified with three sample queries that the index retrieves semantically relevant chunks across different document types (NCDs, LCDs, Benefit Policy manuals).

### Limitations of Dense Retrieval Alone

While the FAISS index provides good semantic matching, it has two key limitations that the next pipeline stage (Person 3) will address:

1. **No state filtering**: A Texas-specific query may retrieve chunks from other states. The retrieval pipeline will add metadata-based filtering using the `states` field.
2. **No reranking**: Bi-encoder retrieval (FAISS) is fast but approximate in relevance judgment. A cross-encoder reranker (`BAAI/bge-reranker-v2-m3`) will rescore the top candidates for higher precision.

### Saved Artifacts for Downstream Use

The following files are saved to Google Drive and will be loaded by the next notebook:

| File | Path |
|---|---|
| FAISS index | `faiss_index/medicare.index` |
| Document IDs | `faiss_index/docids.txt` |
| Raw embeddings | `faiss_index/embeddings.npy` |
| Chunk metadata | `faiss_index/chunk_metadata.json` |

Step 3: Retrieval + Reranking

At this stage, the FAISS index and document embeddings have already been created.

The goal of this step is to:

Convert the user query into an embedding

Retrieve the most similar document chunks using FAISS

Rerank those chunks using a cross-encoder model

In [24]:
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

INDEX_PATH = "medicare.index"
META_PATH = "chunk_metadata.json"
DOCIDS_PATH = "docids.txt"

index = faiss.read_index(INDEX_PATH)

with open(META_PATH) as f:
    metadata = json.load(f)

with open(DOCIDS_PATH) as f:
    docids = [line.strip() for line in f]

with open("all_chunks.json") as f:
    all_chunks = json.load(f)

print("Index loaded")
print("Total chunks:", len(metadata))

Index loaded
Total chunks: 8563


In [25]:
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [27]:
def embed_query(query):

    embedding = embed_model.encode(
        query,
        normalize_embeddings=True
    )

    return embedding.astype("float32")

In [28]:
def retrieve_chunks(query, top_k=20):

    query_vec = embed_query(query)

    scores, indices = index.search(
        np.array([query_vec]),
        top_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        meta = metadata[idx]
        text = all_chunks[idx]["text"]

        results.append({
            "faiss_score": float(scores[0][rank]),
            "text": text,
            "title": meta["title"],
            "type": meta["type"],
            "states": meta["states"],
            "source_id": meta["source_id"]
        })

    return results

In [29]:
def filter_by_state(results, state=None):

    if state is None:
        return results

    filtered = []

    for r in results:

        states = r["states"]

        if "ALL" in states or state in states:
            filtered.append(r)

    return filtered

In [35]:
def rerank_results(query, results, top_n=5):

    pairs = [(query, r["text"]) for r in results]

    scores = reranker.predict(pairs)

    for i in range(len(results)):
        results[i]["rerank_score"] = float(scores[i])

    results.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # remove duplicate documents
    seen = set()
    final = []

    for r in results:

        if r["source_id"] not in seen:
            final.append(r)
            seen.add(r["source_id"])

        if len(final) == top_n:
            break

    return final

In [36]:
def retrieve_and_rerank(query, state=None):

    retrieved = retrieve_chunks(query, top_k=20)

    filtered = filter_by_state(retrieved, state)

    top_chunks = rerank_results(query, filtered, top_n=5)

    return top_chunks

In [37]:
query = "Does Medicare cover acupuncture for chronic lower back pain?"

results = retrieve_and_rerank(query)

for r in results:

    print("TITLE:", r["title"])
    print("TYPE:", r["type"])
    print("SOURCE:", r["source_id"])
    print("RERANK SCORE:", r["rerank_score"])
    print(r["text"][:300])
    print("\n----------------------\n")

TITLE: Acupuncture for Chronic Lower Back Pain (cLBP)
TYPE: NCD
SOURCE: 30.3.3
RERANK SCORE: 0.9996066689491272
[TYPE: NCD | NCD_ID: 30.3.3 | TITLE: Acupuncture for Chronic Lower Back Pain (cLBP)]

TITLE: Acupuncture for Chronic Lower Back Pain (cLBP) SECTION: 30.3.3 EFFECTIVE DATE: 2020-01-21 00:00:00 COVERAGE POLICY: B. Nationally Covered Indications Effective for services performed on or after January 21, 

----------------------

TITLE: Chapter 32 – Billing Requirements for Special Services
TYPE: CLAIMS_PROCESSING
SOURCE: Chapter 32 – Billing Requirements for Special Services
RERANK SCORE: 0.9987831711769104
[TYPE: CLAIMS_PROCESSING | MANUAL: Medicare Claims Processing Manual | CHAPTER: Chapter 32 – Billing Requirements for Special Services | STATES: ALL]

16.77 – This service/item was not covered because it was not provided as part of a qualifying trial/study. Spanish Version – (Este servicio/artículo 

----------------------

TITLE: Acupuncture
TYPE: NCD
SOURCE: 30.3
RERANK SCORE

1. Query Embedding

The query is converted into a dense vector using the same model used during indexing.

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

Example:

Query:
Does Medicare cover acupuncture for chronic lower back pain?

↓

Embedding vector

This ensures the query and document embeddings exist in the same vector space.

2. FAISS Retrieval

The query embedding is passed to FAISS.

FAISS performs a nearest-neighbor search and returns the most similar vectors.

scores, indices = index.search(query_vector, top_k)

We retrieve:

Top 20 candidate chunks

These chunks are semantically similar but may not all be highly relevant.

3. Mapping to Chunk Text

The FAISS index only returns vector IDs.

To get the actual text, we map those IDs using:

chunk_metadata.json → metadata
all_chunks.json → actual chunk text

From this we retrieve:

title
type
source_id
chunk text

4. Cross-Encoder Reranking

Next, we apply a reranker.

Model used:

BAAI/bge-reranker-v2-m3

The reranker evaluates each pair:

(query, chunk)

Example:

(query, chunk_1) → 0.99
(query, chunk_2) → 0.78
(query, chunk_3) → 0.45

Chunks are then sorted by this score.

5. Final Results

After reranking, we keep the top 5 most relevant chunks.

These chunks represent the most relevant Medicare policy sections for the query.

Example output:

TITLE: Acupuncture for Chronic Lower Back Pain
TYPE: NCD
SOURCE: 30.3.3
RERANK SCORE: 0.999
Final Retrieval Flow
User Query
↓
Embedding Model
↓
FAISS Retrieval (20 chunks)
↓
Metadata Mapping
↓
Cross-Encoder Reranking
↓
Top 5 Policy Chunks

These results will be used as context for the final LLM answer generation stage.